In [7]:
import pandas as pd

# ── 1. Lecture Bloomberg ──────────────────────────────────────────────────────
bb = pd.read_csv("bloomberg_data_clean.csv", sep=";", decimal=",")
bb.columns = bb.columns.str.strip()

# Nettoyage des valeurs (espaces dans les nombres -> "2002 11" => "2002.11")
for col in bb.columns[1:]:
    bb[col] = (
        bb[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", ".", regex=True)
        .pipe(pd.to_numeric, errors="coerce")
    )

# Parse date Bloomberg : format "03/01/2000" -> DD/MM/YYYY
bb["date"] = pd.to_datetime(bb["date"].astype(str).str.strip(), format="%d/%m/%Y")

# ── 2. Lecture Fama-French ────────────────────────────────────────────────────
ff = pd.read_csv("fama_french_data.csv", sep=",", skipinitialspace=True)
ff.columns = ff.columns.str.strip()

# Parse date FF : format "19630701" -> YYYYMMDD
ff["Date"] = pd.to_datetime(ff["Date"].astype(str).str.strip(), format="%Y%m%d")

# Renommage des colonnes FF vers les noms cibles
ff = ff.rename(columns={
    "Date":   "date",
    "Mkt-RF": "MKT_RF",
    "SMB":    "SMB",
    "HML":    "HML",
    "RMW":    "RMW",
    "CMA":    "CMA",
})

# On garde uniquement les colonnes utiles (on drop RF)
ff = ff[["date", "MKT_RF", "SMB", "HML", "RMW", "CMA"]]

# ── 3. Merge (left join sur les dates Bloomberg) ──────────────────────────────
merged = bb.merge(ff, on="date", how="left")

# ── 4. Diagnostic : dates Bloomberg sans correspondance FF ───────────────────
missing_mask = merged["MKT_RF"].isna()
missing_dates = merged.loc[missing_mask, "date"]

if missing_dates.empty:
    print("✅ Toutes les dates Bloomberg ont été trouvées dans Fama-French.")
else:
    print(f"⚠️  {len(missing_dates)} date(s) Bloomberg sans données Fama-French :")
    for d in missing_dates:
        print(f"   {d.strftime('%d/%m/%Y')}")

# ── 5. Export ─────────────────────────────────────────────────────────────────
merged["date"] = merged["date"].dt.strftime("%d/%m/%Y")
merged.to_csv("bloomberg_data_clean_v2.csv", sep=";", index=False)

print(f"\n✅ Fichier écrit : bloomberg_data_clean_v2.csv ({len(merged)} lignes)")
print(f"   Colonnes : {list(merged.columns)}")

⚠️  4 date(s) Bloomberg sans données Fama-French :
   11/09/2001
   12/09/2001
   13/09/2001
   14/09/2001

✅ Fichier écrit : bloomberg_data_clean_v2.csv (6543 lignes)
   Colonnes : ['date', 'SPXT', 'LT09TRUU', 'BCOMCOT', 'USDJPY', 'USGG3M', 'MOM', 'MKT_RF', 'SMB', 'HML', 'RMW', 'CMA']
